## Evaluación de K=14

El objetivo de este cuaderno es evaluar si una arquitectura de 14 clústeres permite una mejor especialización de las secciones clínicas o si, por el contrario, resulta en un sobreajuste que penalice la claridad para el paciente. 

Este experimento se realiza sobre el **corpus deduplicado (umbral 0.95)** para asegurar que la segmentación se base en diversidad temática y no en repetición estructural legal.

In [1]:
import pandas as pd
df_parrafos = pd.read_csv("prospectos_unicos_semanticos_15.csv")

df_parrafos = df_parrafos.drop_duplicates(subset=['parrafo_anonimizado'])

### Configuración del Modelo de 14 Tópicos
Se incrementa la k a 14 para observar si es posible separar sub-temas específicos dentro de las reacciones adversas o instrucciones de uso. Buscamos el punto donde la ganancia en especificidad clínica compensa el incremento en la carga cognitiva del usuario.

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Cargar modelo de embeddings (multilingüe para español)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 1. Limpiar y vectorizar
textos = df_parrafos['parrafo_anonimizado'].tolist()
embeddings = model.encode(textos, show_progress_bar=True)

# 2. Clustering
num_clusters = 14 
clustering_model = KMeans(n_clusters=num_clusters, random_state=42)
df_parrafos['cluster'] = clustering_model.fit_predict(embeddings)

c:\Users\04jul\Desktop\CDIA\TFG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 204.41it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 869/869 [17:42<00:00,  1.22s/it]


In [13]:
df_parrafos[df_parrafos["cluster"]==0]["parrafo_anonimizado"]

27       Este medicamento contiene menos de 1 mmol de s...
30       En la prevención de la embolia provocada por u...
40       No tome una dosis doble para compensar las dos...
103      Este medicamento contiene menos de 1 mmol de s...
104      La dosis recomendada de este medicamento para ...
                               ...                        
27736    Adolescentes (a partir de 12 años) y niños ≥ 6...
27737    Segunda dosis: 6,5 mg/kg (sin exceder 400 mg),...
27738    Tercera dosis: 6,5 mg/kg (sin exceder 400 mg),...
27739    Cuarta dosis: 6,5 mg/kg (sin exceder 400 mg), ...
27741    Debe emplearse la dosis mínima eficaz y no deb...
Name: parrafo_anonimizado, Length: 1899, dtype: str

## Preparación de prompt

In [ ]:
def generar_prompt_final(cluster_id, ejemplos):
  ejemplos_str = "\n- ".join(ejemplos)
      
  prompt = f"""[INST] <<SYS>>
  Eres un PACIENTE que tiene una duda urgente y busca la respuesta en estos fragmentos. 
  TU REGLA MÁXIMA: Escribe ÚNICAMENTE la pregunta que harías a tu médico.
  PROHIBIDO usar palabras como: 'texto', 'fragmento', 'prospecto', 'diseño', 'JSON', 'información', 'sección', 'analizar'.
  PROHIBIDO mencionar nombres de medicamentos.
  <</SYS>>
  Sigue este razonamiento interno:
  1. Este fragmento "Puede disminuir la capacidad para conducir o manejar maquinaria por contener propilenglicol, ya que puede producir síntomas parecidos a los del alcohol." responde a la pregunta "¿El paracetamol afecta a la capacidad de conducción?".
  2. Este fragmento "Estas dosis se pueden repetir cada 4 horas." no responde a esa pregunta, responde a "¿Cada cuánto tiempo me lo tengo que tomar?".
  3. Teniendo esto en cuenta, ¿qué PREGUNTA UNIVERSAL responden todos los fragmentos del grupo?
  Basándote en estos fragmentos:
  {ejemplos_str}

  Genera la PREGUNTA que un paciente normal (no experto) haría. 

  Responde exclusivamente en este formato JSON:
  {{
    "pregunta_del_paciente": "Escribe aquí la pregunta"
  }}
  [/INST]"""
  return prompt

## Conexión con Ollama

In [4]:
import requests
import json
import pandas as pd

def consultar_ollama_stream(prompt, modelo_id):
    url = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
    payload = {
        "model": modelo_id,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True
    }
    
    texto_acumulado = ""
    try:
        with requests.post(url, json=payload, stream=True, timeout=300) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line: continue
                decoded = line.decode("utf-8").replace("data: ", "").strip()
                if decoded == "[DONE]": break
                try:
                    chunk = json.loads(decoded)
                    content = chunk["choices"][0].get("delta", {}).get("content", "")
                    texto_acumulado += content
                except: continue
        return texto_acumulado
    except Exception as e:
        return f"ERROR: {str(e)}"

In [17]:
anotadores = ["llama3:latest", "llama3_medium:latest", "llama3_hard:latest"]
resultados = []

In [28]:
import time

for c_id in sorted(df_parrafos['cluster'].unique()):
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    ejemplos_cortos = [texto[:500] + "..." if len(texto) > 500 else texto for texto in ejemplos_variados]
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_cortos)
    print(len(ejemplos_variados))
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_cortos)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) # Pausa para no saturar el servidor

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("resultados_etiquetado_14_topicos.csv", index=False)

 Procesando Cluster 0...
1899
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "¿Qué dosis recomienda para mi caso específico?"
}
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Qué dosis puedo tomar si tengo sobrepeso?"
}
  -> Consultando llama3_hard:latest...
{
  "pregunta_del_paciente": "¿Cuál es el límite de dosis al día para adultos?"
}
 Procesando Cluster 1...
1552
  -> Consultando llama3:latest...
{
"pregunta_del_paciente": "Puedo seguir tomando [MEDICAMENTO] si me olvido un comprimido y no tengo regla durante el intervalo de placebo?"
}
  -> Consultando llama3_medium:latest...
{
"pregunta_del_paciente": "¿Puedo seguir tomando [MEDICAMENTO] si estoy embarazada o intentando quedarme embarazada?"
}
  -> Consultando llama3_hard:latest...
{
  "pregunta_del_paciente": "Puedo tomar [MEDICAMENTO] si estoy embarazada o en período de lactancia?"
}
 Procesando Cluster 2...
1908
  -> Consultando llama3:latest...
{
  "pregunta_del_paciente": "¿Cuanto tiempo

In [ ]:
import pandas as pd
import json
import re

def extraer_pregunta_segura(texto):
    """
    Limpia y extrae la pregunta del JSON generado por el LLM, 
    gestionando texto basura y fallos de formato.
    """
    if pd.isna(texto) or "ERROR" in str(texto):
        return "Error en respuesta"
    
    match = re.search(r'\{.*\}', str(texto), re.DOTALL)
    
    if match:
        json_str = match.group()
        try:
            # Reemplazar comillas dobles repetidas
            json_str = json_str.replace('""', '"')
            datos = json.loads(json_str)

            pregunta = datos.get('pregunta_final')
            
            if pregunta:
                return pregunta.strip()
        except Exception:
            pass

    # Si falla el JSON, buscamos patrones de texto
    # Busca frases que empiecen por "¿" y terminen por "?"
    preguntas_encontradas = re.findall(r'¿[^?]+\?', str(texto))
    if preguntas_encontradas:
        # Devolvemos la última pregunta encontrada 
        return preguntas_encontradas[-1].strip()

    return "No se pudo identificar la pregunta"




df_resultados = pd.read_csv('resultados_etiquetado_14_topicos.csv')


anotadores = [col for col in df_resultados.columns if col.startswith('res_')]

for col in anotadores:
    # Creamos una columna limpia para cada modelo
    nombre_limpio = col.replace('res_', 'pregunta_')
    df_resultados[nombre_limpio] = df_resultados[col].apply(extraer_pregunta_segura)


columnas_formulario = ['cluster'] + [col for col in df_resultados.columns if col.startswith('pregunta_')]
print(df_resultados[columnas_formulario].head())


df_resultados[columnas_formulario].to_csv('preguntas_para_formulario_14.csv', index=False)

   cluster                             pregunta_llama3_latest  \
0        0  ¿Qué hago si me olvido una dosis de [MEDICAMEN...   
1        1  ¿Qué hago si olvido un comprimido y estoy emba...   
2        2  ¿Qué hago si me olvido una dosis de [MEDICAMEN...   
3        3  ¿Qué hago si siento dolor o enrojecimiento en ...   
4        4  ¿Qué hago si siento mareo o dolor de cabeza de...   

                       pregunta_llama3_medium_latest  \
0      ¿Qué hago si accidentalmente carga una dosis?   
1  ¿Es peligroso para el bebé si estoy embarazada...   
2  ¿Qué hago si me olvido una dosis y ya es hora ...   
3                                     ¿Es peligroso?   
4  ¿Por qué hay tantos efectos secundarios peligr...   

                         pregunta_llama3_hard_latest  
0  ¿Y qué sucede si el indicador de dosis muestra...  
1                                   ¿qué debo hacer?  
2                            ¿Debo tomarla de nuevo?  
3     ¿debo parar de tomarlo o hablar con mi médico?

La evaluación de las propuestas del comité de Llama3 (Latest, Medium, Hard) revela un patrón claro de sobreajuste. Al forzar 14 grupos, el algoritmo ha dividido áreas semánticas que deberían estar unidas, rompiendo el consenso de los modelos:

####  El Colapso por Sobreajuste
* **Efectos Secundarios Rotos (Clusters 3, 4, 8):** El algoritmo ha partido las reacciones adversas en pedazos incomprensibles. En el *Cluster 3*, los modelos divagan sobre "acné" y "lentes de contacto". En el *Cluster 4* hablan de "sobredosis y náuseas", y en el *Cluster 8* intentan resumir "peligros graves". El paciente no sabría en qué sección buscar un efecto secundario.
* **Confusión de Formatos (Cluster 5 y 9):** El *Cluster 5* mezcla el "uso del anillo", "olvidar un comprimido rosa" y "tomar el gel", lo que indica que el agrupamiento se basó en palabras irrelevantes y no en la intención. El *Cluster 9* mezcla "mariconsultoría" (alucinación) con el uso de "alcohol" y "cremas para la piel".
* **Falsa Caducidad (Cluster 2):** Llama3_latest interpreta que habla de caducidad, *medium* de duración del tratamiento y *hard* de si lo toma correctamente. Hay una divergencia total.

#### Las Ganancias Locales
A pesar del colapso general, esta alta resolución logró aislar un bloque crítico:
* **Pediatría (Cluster 12):** Hay un consenso total y perfecto. Los tres modelos detectan unánimemente que la intención es saber si el fármaco se puede usar en niños ("¿Puedo usar [MEDICAMENTO] si tengo menos de 18 años?").

## Evaluación de diferentes modelos
Ante la evidente fragmentación semántica y falta de consenso observada con Llama3 (8B), surge una pregunta de investigación: ¿El fracaso de la arquitectura de 14 tópicos se debe al sobreajuste topológico de los datos o a una limitación de capacidad del modelo LLM utilizado?

Para aislar la causa, se diseña un experimento de control utilizando un **Comité de Modelos de Gran Escala**:
* **DeepSeek-R1 (32B):** Modelo avanzado con capacidades de razonamiento.
* **Llama-3.1 (70B):** Uno de los modelos *open-weights* más potentes del mundo.
* **Gemma-3 (27B):** Modelo de última generación de Google.

Se les proporciona un *prompt* base simplificado para evaluar si su mayor cantidad de parámetros les permite encontrar el consenso en un espacio de alta resolución ($K=14$).

In [ ]:
import requests
import json
import pandas as pd
import time


URL_BASE = "https://wiig.dia.fi.upm.es/ollama/v1/chat/completions"
COMITE_MODELOS = ["deepseek-r1:32b", "llama3.1:70b", "gemma3:27b"]

def consultar_modelo(modelo, prompt):
    """Realiza la petición al endpoint y limpia la respuesta."""
    payload = {
        "model": modelo,
        "messages": [
            {"role": "system", "content": "Eres un experto en lingüística médica. Responde exclusivamente con la pregunta solicitada, sin introducciones."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.1
    }
    
    try:
        response = requests.post(URL_BASE, json=payload, timeout=500)
        response.raise_for_status()
        content = response.json()['choices'][0]['message']['content'].strip()
        
        if "</think>" in content:
            content = content.split("</think>")[-1].strip()
            
        return content
    except Exception as e:
        return f"ERROR: {str(e)}"

def procesar_experimento_consenso(df_clusters, columna_texto='parrafo_anonimizado'):
    """
    Agrupa por cluster, toma una muestra de párrafos representativos 
    y consulta al comité de modelos.
    """
    resultados_finales = []
    
    # Agrupamos por cluster ID
    clusters = df_clusters['cluster'].unique()
    
    for cluster_id in sorted(clusters):
        print(f"\n--- Procesando Cluster: {cluster_id} ---")
        
        # Tomamos los párrafos de este cluster (limitamos a los primeros para no saturar el prompt)
        textos = df_clusters[df_clusters['cluster'] == cluster_id][columna_texto].head(10).tolist()
        contexto_cluster = " ".join(textos)
        
        prompt_gen = f"A partir de estos fragmentos de un prospecto médico, genera una pregunta breve que un paciente haría para obtener esta información: {contexto_cluster}"
        
        fila_resultado = {'cluster_id': cluster_id}
        
        for mod in COMITE_MODELOS:
            print(f"  > Consultando a {mod}...")
            pregunta = consultar_modelo(mod, prompt_gen)
            print(pregunta)
            fila_resultado[f'pregunta_{mod.split(":")[0]}'] = pregunta
            
        resultados_finales.append(fila_resultado)
        
    return pd.DataFrame(resultados_finales)


df_consenso = procesar_experimento_consenso(df_parrafos)

# Guardamos el resultado
nombre_archivo = "consenso_preguntas_clusters_14top_modelos_nuevos.csv"
df_consenso.to_csv(nombre_archivo, index=False, encoding='utf-8')

print(df_consenso.head())


--- Procesando Cluster: 0 ---
  > Consultando a deepseek-r1:32b...
¿Este medicamento es bajo en sodio o exento de sodio?
  > Consultando a llama3.1:70b...
¿Cuánto sodio hay en cada dosis del medicamento?
  > Consultando a gemma3:27b...
¿Cuánto sodio contiene este medicamento por dosis?

--- Procesando Cluster: 1 ---
  > Consultando a deepseek-r1:32b...
¿Es seguro tomar este medicamento durante el embarazo o la lactancia y qué efectos podría tener en el embrión, feto o bebé?
  > Consultando a llama3.1:70b...
¿Es seguro tomar este medicamento durante el embarazo o la lactancia?
  > Consultando a gemma3:27b...
¿Puedo tomar este medicamento si estoy embarazada, planeo quedar embarazada o estoy amamantando?

--- Procesando Cluster: 2 ---
  > Consultando a deepseek-r1:32b...
¿Hasta cuándo puedo usar este medicamento de manera segura?
  > Consultando a llama3.1:70b...
¿Cuándo y cómo debo tomar este medicamento?
  > Consultando a gemma3:27b...
¿Cómo y cuándo debo tomar este medicamento, y qué

Los resultados del comité  confirman de manera irrefutable que **el problema reside en la arquitectura de la información (K=14), no en el tamaño de la red neuronal**. El exceso de fragmentación estresa a los modelos pesados, provocando anomalías:

1. **Ruptura de Instrucciones (System Prompt Override):** DeepSeek-R1 (32B), diseñado para ser analítico, ignora la instrucción de responder exclusivamente con la pregunta y añade meta-comentarios en inglés (*"The appropriate question for a patient seeking information on when not to take the medication is..."* en el Cluster 6).
2. **Colapso Generativo (Token Degradation):** El fallo más crítico se observa en Gemma-3 (27B). Al enfrentarse a clústeres de información residual o sobre-fragmentada (Clusters 11 y 12), el modelo colapsa, perdiendo el anclaje lingüístico y emitiendo un bucle de caracteres en hindi, árabe y ruido alfanumérico (`मुख्यमंत्रीکنमारीdcमारीdcdcdc...`). Esto es un síntoma técnico de que el contexto carece de la cohesión semántica mínima necesaria para que el mecanismo de atención del *Transformer* funcione.

## Evaluación de modelos con más capacidad con prompt detallado

Tras demostrar empíricamente que los modelos colapsan (degradación de tokens) o alucinan (sesgo de especificidad) al enfrentarse a textos médicos fragmentados con un *prompt* simple, procedemos a evaluar estos mismos modelos con el *prompt* más extenso y detallado que usamos en la primera prueba de este notebook.

Se implementa una arquitectura de *prompting* combinando tres técnicas del estado del arte:
1. **Role-Play Cognitivo:** Posicionar al LLM como "Paciente no experto".
2. **Restricciones Negativas Estrictas (Negative Constraints):** Prohibición explícita de usar metadatos (ej. "fragmento", "JSON") o nombres de medicamentos para forzar la abstracción.
3. **Razonamiento Guiado (Chain of Thought / Few-Shot):** Provisión de ejemplos lógicos para enseñar al modelo a diferenciar entre el texto literal y la intención subyacente.

In [ ]:
import time

anotadores = ["deepseek-r1:32b", "llama3_hard:latest", "gemma3:27b"]
resultados = []

for c_id in sorted(df_parrafos['cluster'].unique()):
    print(f" Procesando Cluster {c_id}...")
    
    # 1. Selección de variedad
    df_c = df_parrafos[df_parrafos['cluster'] == c_id]
    ejemplos_variados = df_c["parrafo_anonimizado"].to_list()
    ejemplos_cortos = [texto[:500] + "..." if len(texto) > 500 else texto for texto in ejemplos_variados]
    
    # 2. Generación del prompt
    prompt_texto = generar_prompt_final(c_id, ejemplos_cortos)
    
    fila = {"cluster": c_id, "ejemplos_usados": "|".join(ejemplos_cortos)}
    
    # 3. Consulta a los LLMs
    for modelo in anotadores:
        print(f"  -> Consultando {modelo}...")
        respuesta = consultar_ollama_stream(prompt_texto, modelo)
        print(respuesta)
        fila[f"res_{modelo.replace(':', '_')}"] = respuesta
        time.sleep(1) 

    resultados.append(fila)

df_final = pd.DataFrame(resultados)
df_final.to_csv("preguntas_v2_14topicos.csv", index=False)

 Procesando Cluster 0...
  -> Consultando deepseek-r1:32b...
```json
{
    "pregunta_del_paciente": "¿Cuál es la dosis correcta que debo tomar?"
}
```
  -> Consultando llama3_hard:latest...
{
"pregunta_del_paciente": "¿Cuál es la dosis recomendada para mi caso después de accidentalmente tomar un exceso de [MEDICAMENTO]? "
}
  -> Consultando gemma3:27b...
```json
{
  "pregunta_del_paciente": "¿Cómo debo tomar este medicamento y cada cuánto tiempo?"
}
```
 Procesando Cluster 1...
  -> Consultando deepseek-r1:32b...
{
    "pregunta_del_paciente": "¿Puedo quedarme embarazada mientras estoy tomando este medicamento?"
}
  -> Consultando llama3_hard:latest...
{
  "pregunta_del_paciente": "¿Qué pasa si olvido tomar [MEDICAMENTO]? ¿Puedo retrasar mi período menstrual y seguir tomando el medicamento como si fuera un nuevo envase?"
}
  -> Consultando gemma3:27b...
```json
{
  "pregunta_del_paciente": "¿Qué riesgos tiene este medicamento si estoy embarazada, planeo quedar embarazada o estoy dando 

### Evaluación del Comité y Selección del Modelo Definitivo
La ejecución del nuevo *prompt* sobre el comité de modelos revela resultados altamente concluyentes que definen la arquitectura final del proyecto:

1. **Llama3_hard (8B) - Sobreajuste Persistente:** A pesar de las instrucciones, el modelo menor sigue siendo víctima de los datos. En el Cluster 13, asume erróneamente que el contexto trata sobre tratar la malaria con hidroxicloroquina. Carece de la capacidad de abstracción necesaria.
2. **DeepSeek-R1 (32B) - Inestabilidad de Infraestructura y Alineación:** El modelo sufre múltiples Errores 500 debido a la carga computacional en el servidor. Además, en el Cluster 7 presenta un fallo de alineación multilingüe (*Token Bleed*), mezclando español con el término árabe "الجانب" (Al-Janib, traducción literal de "aspecto/lado" en el contexto de "efectos secundarios").
3. **Gemma-3 (27B)** El modelo de Google interpreta perfectamente el *Chain of Thought*. Logra despojarse del sesgo léxico y genera Categorías Universales precisas, seguras y formuladas en lenguaje natural de paciente (ej. *"¿Qué riesgos tiene este medicamento si estoy embarazada, planeo quedar embarazada o estoy dando el pecho?"*). Su única desviación es añadir etiquetas de formato Markdown, fácilmente limpiables en post-procesamiento.

Los resultados de esta iteración nos permiten extraer una conclusión arquitectónica: **incrementar el número de clústeres no mejora la calidad de la información, sino que la fragmenta**. 

A pesar de contar con un *prompt* robusto y LLMs de última generación, las preguntas resultantes son demasiado específicas. Para un paciente, tener un índice con 14 preguntas donde muchas se solapan temáticamente (ej. varias preguntas distintas solo para efectos secundarios) supone una carga cognitiva notable.

Por lo tanto, **dejamos atrás la configuración de 14 clústeres para probar con un número menor ($K=12$)**. El objetivo de reducir la *k* será forzar al algoritmo a "fusionar" estas preguntas tan específicas en categorías universales más amplias, útiles y representativas del conjunto de datos.

In [9]:
df_parrafos.to_csv("prospectos_clusters14.csv", index=False)